In [1]:
PRODUCTS = {
    "notebook": 60,
    "pen": 10,
    "pencil": 5,
    "school bag": 500,
    "eraser": 5
}

SYSTEM = """
You are a helpful assistant for Campus Stationery Store.

RULES:
1. Greet customers politely.
2. Answer questions about stationery products.
3. Give product prices.
4. Help customers calculate order totals.
5. Remember customer details during the conversation.
6. Do not invent product prices.
"""

def get_price(item: str, qty: int = 1) -> int:
    """Calculate the total price of a stationery item."""
    item = item.lower()

    if item not in PRODUCTS:
        return 0

    return PRODUCTS[item] * qty


# Test the function
total = get_price("notebook", 5)
print("Total for 5 notebooks:", total)


# Chatbot setup
history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8


def chat(user_message):
    global history

    history.append({"role": "user", "content": user_message})

    recent = [history[0]] + history[-MAX_TURNS * 2:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content
        history.append({"role": "assistant", "content": reply})

        return reply

    except Exception as e:
        return f"Error: {e}"


print("\nCampus Stationery Store Chatbot Ready!")

while True:
    text = input("You: ")

    if text.lower() in ["bye", "exit", "quit"]:
        print("Bot: Thank you for visiting Campus Stationery Store!")
        break

    print("Bot:", chat(text))

Total for 5 notebooks: 300

Campus Stationery Store Chatbot Ready!
You: Hi, my name is Ravi
Bot: Error: name 'client' is not defined
You: bye
Bot: Thank you for visiting Campus Stationery Store!


In [4]:
!pip install -q langchain langchain-openai

In [5]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

In [6]:
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_KEY2"),
    base_url="https://api.groq.com/openai/v1",
    temperature=0.7
)

print("Groq connected successfully!")

Groq connected successfully!


In [7]:
SYSTEM = """ROLE:
You are a friendly and helpful AI assistant for Campus Stationery Store.

FACTS:
Notebook = Rs 60.
Pen = Rs 10.
Pencil = Rs 5.
School Bag = Rs 500.
Eraser = Rs 5.
Orders need 15 minutes to prepare.

RULES:
1. Always answer using the given FACTS.
2. When asked about a product price, mention the exact price.
3. Never invent prices or product details.
4. If information is unavailable, say "I'll check with the store staff."
5. Calculate quantity totals correctly.
6. Never promise an exact delivery time.
7. Do not offer discounts unless included in FACTS.

FORMAT:
Under 50 words.
Friendly English or Telugu matching the customer.
"""

print("System prompt created successfully!")

System prompt created successfully!


In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

print("Prompt and chain created successfully!")

Prompt and chain created successfully!


In [9]:
tests = [
    "What products are available?",
    "What is the price of a notebook?",
    "What is the price of 5 pens?",
    "Do you have a laptop?",
    "How long do orders take to prepare?"
]

def run_tests():
    for question in tests:
        answer = chain.invoke({
            "history": [],
            "question": question
        })

        print("Question:", question)
        print("Answer:", answer)
        print("-" * 50)

print("Test function created successfully!")

Test function created successfully!


In [10]:
print("Testing Campus Stationery Store Assistant:")
run_tests()

Testing Campus Stationery Store Assistant:
Question: What products are available?
Answer: We have notebooks (Rs 60), pens (Rs 10), pencils (Rs 5), school bags (Rs 500) and erasers (Rs 5). All items are ready to order—just let us know what you need!
--------------------------------------------------
Question: What is the price of a notebook?
Answer: A notebook costs Rs 60.
--------------------------------------------------
Question: What is the price of 5 pens?
Answer: The price of 5 pens is Rs 50.
--------------------------------------------------
Question: Do you have a laptop?
Answer: I’ll check with the store staff.
--------------------------------------------------
Question: How long do orders take to prepare?
Answer: It takes about 15 minutes to prepare your order.
--------------------------------------------------


In [11]:
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_KEY2"),
    base_url="https://api.groq.com/openai/v1",
    temperature=1.0
)

chain = prompt | llm | StrOutputParser()

print("Temperature changed to 1.0!")

Temperature changed to 1.0!


In [12]:
print("First Run:")
run_tests()

print("\nSecond Run:")
run_tests()

First Run:
Question: What products are available?
Answer: Here’s what we have in stock:  
Notebook, Pen, Pencil, School Bag, and Eraser.  
Let me know what you’d like to order!
--------------------------------------------------
Question: What is the price of a notebook?
Answer: The price of a notebook is Rs 60.
--------------------------------------------------
Question: What is the price of 5 pens?
Answer: The price of 5 pens is Rs 50.
--------------------------------------------------
Question: Do you have a laptop?
Answer: I’m sorry, I don’t have that information. I’ll check with the store staff and let you know.
--------------------------------------------------
Question: How long do orders take to prepare?
Answer: Orders take 15 minutes to prepare.
--------------------------------------------------

Second Run:
Question: What products are available?
Answer: Here are our items: Notebook (Rs 60), Pen (Rs 10), Pencil (Rs 5), School Bag (Rs 500), Eraser (Rs 5). Let me know if you’d li

In [13]:
jobs = [
    {
        "job": "Calculate the total price of 5 notebooks",
        "type": "Plain Code"
    },
    {
        "job": "Answer customer questions about stationery products",
        "type": "One LLM Call"
    },
    {
        "job": "Check stationery stock availability using a store tool",
        "type": "Agent"
    }
]

for item in jobs:
    print("Job:", item["job"])
    print("Type:", item["type"])
    print()

Job: Calculate the total price of 5 notebooks
Type: Plain Code

Job: Answer customer questions about stationery products
Type: One LLM Call

Job: Check stationery stock availability using a store tool
Type: Agent



In [14]:
customer_questions = [
    "Hi, my name is Ravi.",
    "What products are available?",
    "What is the price of a notebook?",
    "What is the price of 5 pens?",
    "Do you have a laptop?",
    "How long do orders take to prepare?",
    "What is my name?"
]

for question in customer_questions:
    answer = chain.invoke({
        "history": [],
        "question": question
    })

    print("Customer:", question)
    print("Assistant:", answer)
    print("-" * 50)

Customer: Hi, my name is Ravi.
Assistant: Hi Ravi! 👋 How can I help you at Campus Stationery Store today?
--------------------------------------------------
Customer: What products are available?
Assistant: We have notebooks, pens, pencils, school bags, and erasers.  
Prices: Notebook Rs 60, Pen Rs 10, Pencil Rs 5, School Bag Rs 500, Eraser Rs 5.
--------------------------------------------------
Customer: What is the price of a notebook?
Assistant: A notebook is Rs 60.
--------------------------------------------------
Customer: What is the price of 5 pens?
Assistant: 5 pens cost ₹10 each, so the total is ₹50.
--------------------------------------------------
Customer: Do you have a laptop?
Assistant: I’m not sure—our store mainly sells stationery. I’ll check with the staff and let you know!
--------------------------------------------------
Customer: How long do orders take to prepare?
Assistant: Orders take about 15 minutes to prepare. Let me know what you’d like to order!
--------